<a href="https://colab.research.google.com/github/rendyamril/data-science-2026/blob/main/Pertemuan12_MohammadRendyAmril_24040101028.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

* Nama : Mohammad Rendy Amril
* NIM : 240401010288
* Prodi : S1 PJJ Informatika
* Kelas : IF 403

In [8]:
import warnings
warnings.filterwarnings('ignore')

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [1]:
!pip install -q mlxtend

In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
produk = ['Roti', 'Selai', 'Susu', 'Sereal', 'Telur',
          'Keju', 'Kopi', 'Gula', 'Teh', 'Mentega']

# Buat 50 transaksi, tiap transaksi berisi 2-5 produk
transaksi = []
for i in range(50):
    n_item = np.random.randint(2, 6)
    transaksi.append(list(np.random.choice(produk, n_item, replace=False)))

# Suntikkan pola khusus: Roti sering dibeli bersama Selai
for i in range(0, 20):
    if 'Roti' in transaksi[i] and 'Selai' not in transaksi[i]:
        transaksi[i].append('Selai')

print('Contoh 3 transaksi pertama:', transaksi[:3])
print('Total transaksi:', len(transaksi))

Contoh 3 transaksi pertama: [[np.str_('Keju'), np.str_('Roti'), np.str_('Mentega'), np.str_('Kopi'), 'Selai'], [np.str_('Roti'), np.str_('Kopi'), np.str_('Teh'), np.str_('Selai'), np.str_('Mentega')], [np.str_('Kopi'), np.str_('Susu'), np.str_('Teh')]]
Total transaksi: 50


In [10]:
from mlxtend.preprocessing import TransactionEncoder

te = TransactionEncoder()
te_ary = te.fit(transaksi).transform(transaksi)
df = pd.DataFrame(te_ary, columns=te.columns_)

print("Tampilan awal data transaksi (One-Hot Encoded):")
df.head()

Tampilan awal data transaksi (One-Hot Encoded):


,Gula,Keju,Kopi,Mentega,Roti,Selai,Sereal,Susu,Teh,Telur
0,False,True,True,True,True,True,False,False,False,False
1,False,False,True,True,True,True,False,False,True,False
2,False,False,True,False,False,False,False,True,True,False
3,False,True,False,False,False,True,False,False,True,True
4,True,True,False,True,False,False,False,True,False,False


In [11]:
from mlxtend.frequent_patterns import apriori

# Eksplorasi beberapa nilai min_support
for ms in [0.05, 0.1, 0.2]:
    freq = apriori(df, min_support=ms, use_colnames=True)
    print(f'min_support={ms}: {len(freq)} itemset ditemukan')

# Gunakan min_support = 0.1 untuk analisis selanjutnya
freq_items = apriori(df, min_support=0.1, use_colnames=True)
freq_items = freq_items.sort_values('support', ascending=False)

print("\nTop 10 Frequent Itemsets:")
freq_items.head(10)

min_support=0.05: 74 itemset ditemukan
min_support=0.1: 44 itemset ditemukan
min_support=0.2: 13 itemset ditemukan

Top 10 Frequent Itemsets:


,support,itemsets
5,0.52,(Selai)
8,0.46,(Teh)
3,0.42,(Mentega)
9,0.36,(Telur)
1,0.34,(Keju)
0,0.32,(Gula)
2,0.32,(Kopi)
4,0.32,(Roti)
7,0.32,(Susu)
36,0.24,"(Selai, Teh)"


In [12]:
from mlxtend.frequent_patterns import association_rules

rules = association_rules(freq_items, metric='confidence', min_threshold=0.5)
rules = rules[rules['lift'] > 1].sort_values('lift', ascending=False)

print("Top Aturan Asosiasi berdasarkan Nilai Lift Tertinggi:")
rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10)

Top Aturan Asosiasi berdasarkan Nilai Lift Tertinggi:


,antecedents,consequents,support,confidence,lift
9,"(Keju, Teh)",(Telur),0.12,0.857143,2.380952
15,"(Mentega, Selai)",(Kopi),0.10,0.625000,1.953125
11,"(Roti, Gula)",(Selai),0.10,1.000000,1.923077
7,(Sereal),(Mentega),0.14,0.777778,1.851852
10,"(Telur, Teh)",(Keju),0.12,0.600000,1.764706
14,"(Kopi, Selai)",(Mentega),0.10,0.714286,1.700680
8,"(Keju, Telur)",(Teh),0.12,0.750000,1.630435
12,"(Selai, Gula)",(Roti),0.10,0.500000,1.562500
13,"(Kopi, Mentega)",(Selai),0.10,0.714286,1.373626
1,(Roti),(Selai),0.22,0.687500,1.322115


In [13]:
from sklearn.metrics.pairwise import cosine_similarity

# Katalog Produk Sintetis
katalog = pd.DataFrame({
    'produk': produk,
    'kategori': ['Bakery', 'Bakery', 'Dairy', 'Bakery', 'Dairy',
                 'Dairy', 'Minuman', 'Bumbu', 'Minuman', 'Dairy']
})

# Feature extraction dari kategori
fitur = pd.get_dummies(katalog['kategori'])
sim_matrix = cosine_similarity(fitur)

def rekomendasi_serupa(nama_produk, top_n=3):
    idx = katalog.index[katalog['produk'] == nama_produk][0]
    skor = list(enumerate(sim_matrix[idx]))
    skor = sorted(skor, key=lambda x: x[1], reverse=True)
    skor = [s for s in skor if s[0] != idx][:top_n]
    return katalog.iloc[[i for i, _ in skor]]['produk'].tolist()

print('Produk serupa dengan Roti (Content-Based):', rekomendasi_serupa('Roti'))

Produk serupa dengan Roti (Content-Based): ['Selai', 'Sereal', 'Susu']


In [14]:
produk_target = 'Roti'

# Dari Association Rules (Market Basket Analysis)
rules_terkait = rules[rules['antecedents'].apply(lambda x: produk_target in x)]

print(f"=== REKOMENDASI UNTUK PRODUK: {produk_target} ===")
print("\n1. Rekomendasi dari Association Rules (Berdasarkan Kebiasaan Belanja):")
print(rules_terkait[['consequents', 'lift']].head())

print("\n2. Rekomendasi dari Content-Based Filtering (Berdasarkan Kesamaan Kategori):")
print(rekomendasi_serupa(produk_target))

=== REKOMENDASI UNTUK PRODUK: Roti ===

1. Rekomendasi dari Association Rules (Berdasarkan Kebiasaan Belanja):
   consequents      lift
11     (Selai)  1.923077
1      (Selai)  1.322115

2. Rekomendasi dari Content-Based Filtering (Berdasarkan Kesamaan Kategori):
['Selai', 'Sereal', 'Susu']



**Kesimpulan**
Modul ini membahas Association Rule Mining menggunakan algoritma Apriori untuk menemukan pola transaksi (Market Basket Analysis) melalui metrik Support, Confidence, dan Lift. Selain itu, modul mengenalkan dasar Sistem Rekomendasi melalui pendekatan Collaborative Filtering, Content-Based Filtering, serta pengujian kualitasnya memakai metrik Precision@K.

**Temuan Utama**
* Nilai Lift lebih dari 1 menunjukkan korelasi antar produk yang benar-benar kuat.
* Precision@K mengukur proporsi item relevan pada Top-K rekomendasi.

**Pertanyaan yang Muncul**
Bagaimana menentukan batas nilai minimum support dan confidence yang paling presisi agar aturan yang terbentuk tidak terlalu banyak atau terlalu sedikit?